# Notebook A: MRI scans to harmonized training slices

Step 1 of the Kaggle pipeline. Builds the real-scan dataset and saves it as this notebook's output folder `data_real/`:

```
data_real/Training/<class>/<patientID>_z###.jpg   (_y### coronal, _x### sagittal)
data_real/Testing/<class>/<patientID>_z###.jpg
data_real/manifest.csv                           source dataset of every slice
```

Each patient goes entirely into Training or Testing (20% of patients are held out). The conversion code lives in `nifti_slices.py` and `harmonize.py` in the repo.

**Why harmonized:** every class used to come from its own dataset, and the models learned to recognise the dataset (brightness scale, skull, resolution) instead of the tumor. Every source now gets the same orientation, 1 mm resampling, skull stripping, per-volume intensity normalization and field of view, and glioma and meningioma get a second source (Figshare).

**Set up once:**
1. *Add Input*: `vietanh21/brats-2021-task-1-dataset`, `dukhailly/meningioma` and `ashkhagan/figshare-brain-tumor-dataset` (the second glioma/meningioma source; set `USE_FIGSHARE = False` to skip it).
2. Settings: **Internet on**, **Accelerator: GPU T4** (HD-BET skull stripping runs on the GPU; on CPU it takes hours. Set `SKULLSTRIP = "heuristic"` to run without a GPU, with rougher skull stripping).

**Run:** **Save Version > Save & Run All (Commit)**. Notebook B (`train_kaggle.ipynb`) adds this version's output as its input. Then look at the preview grid: every row should look alike apart from the tumor.

`MAKE_ZIP` also writes `data_real.zip`, only needed if you still train on Colab.

In [ ]:
# ---------- Build data_real/ (this one cell does the whole job) ----------
import glob, os, shutil, subprocess, sys
from IPython.display import FileLink, display

OUT_DIR = "/kaggle/working/data_real"
ZIP_BASE = "/kaggle/working/data_real"          # .zip is added automatically
MAKE_ZIP = False                                # only needed for the Colab notebook
REPO_DIR = "/tmp/repo"                          # outside /kaggle/working, so it is not saved as output
BRANCH = "main"
SLICES_PER_PATIENT = 8
MAX_PATIENTS = 400

# Harmonization (see harmonize.py)
SKULLSTRIP = "hd-bet"          # "hd-bet" (GPU), "heuristic" (rough, CPU only), or "none"
INTENSITY = "zscore"           # "zscore" or "histmatch"
PLANES = ["axial", "coronal", "sagittal"]   # Figshare slices mix all three views
USE_FIGSHARE = True            # second source for glioma + meningioma
INCLUDE_METASTASIS = True      # downloads ~8.5 GB from Figshare into /tmp
EXTRA_ARGS = ["--notumor_from_tumor_patients"]   # notumor slices from BraTS scanners too


def find_input(slug, required=True):
    """Find an attached dataset folder by its slug, wherever Kaggle mounted it."""
    hits = [d for d in glob.glob("/kaggle/input/**/", recursive=True)
            if os.path.basename(d.rstrip("/")) == slug and d.count("/") <= 7]
    if not hits:
        if required:
            raise FileNotFoundError(
                f"Dataset '{slug}' is not attached. Add it with 'Add Input' on the right, then re-run.")
        return None
    return sorted(hits, key=len)[0].rstrip("/")


def run(cmd):
    """Run a shell command, show its output live, and stop if it fails."""
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


# 1. Datasets
GLIOMA_ROOT = find_input("brats-2021-task-1-dataset")
MENINGIOMA_ROOT = find_input("meningioma")
FIGSHARE_ROOT = find_input("figshare-brain-tumor-dataset", required=USE_FIGSHARE) if USE_FIGSHARE else None
print("glioma:    ", GLIOMA_ROOT)
print("meningioma:", MENINGIOMA_ROOT)
print("figshare:  ", FIGSHARE_ROOT)

# 2. Code and tools
run(["rm", "-rf", REPO_DIR])
run(["git", "clone", "--depth", "1", "-b", BRANCH,
     "https://github.com/imfonggg/AI-for-Brain-Tumor-Detection.git", REPO_DIR])
run([sys.executable, "-m", "pip", "install", "-q", "nibabel", "h5py"])
if SKULLSTRIP == "hd-bet":
    run([sys.executable, "-m", "pip", "install", "-q", "hd-bet"])

# 3. Convert (takes a while: IXI healthy scans are streamed over the internet)
run(["rm", "-rf", OUT_DIR])
cmd = [sys.executable, "-u", f"{REPO_DIR}/nifti_slices.py", "--out_dir", OUT_DIR,
       "--glioma_root", GLIOMA_ROOT,
       "--meningioma_root", MENINGIOMA_ROOT,
       "--ixi",
       "--skullstrip", SKULLSTRIP, "--intensity", INTENSITY, "--planes", *PLANES,
       "--slices_per_patient", str(SLICES_PER_PATIENT),
       "--max_patients", str(MAX_PATIENTS),
       *EXTRA_ARGS]
if FIGSHARE_ROOT:
    cmd += ["--figshare_root", FIGSHARE_ROOT]
if INCLUDE_METASTASIS:
    cmd += ["--download_metastasis"]
run(cmd)
n_images = len(glob.glob(f"{OUT_DIR}/*/*/*.jpg"))
if n_images == 0:
    raise RuntimeError("No slices were written. Check the log above for the class that found 0 cases.")
print(f"\nOK: {n_images} slices in {OUT_DIR}")

# 4. Zip (optional, for Colab only)
if MAKE_ZIP:
    zip_path = shutil.make_archive(ZIP_BASE, "zip", OUT_DIR)
    print(f"Saved {zip_path} ({os.path.getsize(zip_path) / 1e6:.0f} MB)")
    os.chdir("/kaggle/working")
    display(FileLink(os.path.basename(zip_path)))
else:
    print("data_real/ is ready. After Save Version, add this notebook's output to notebook B as an input.")

In [ ]:
# ---------- Preview 6 random slices per source and class ----------
# Every row should look alike (same brightness, no skull, same size); only the tumor should differ.
import csv, os, random
import matplotlib.pyplot as plt
from PIL import Image

OUT_DIR = "/kaggle/working/data_real"
rows = list(csv.DictReader(open(f"{OUT_DIR}/manifest.csv")))
groups = {}
for r in rows:
    groups.setdefault((r["source"], r["class"]), []).append(r["path"])
keys = sorted(groups)
fig, axes = plt.subplots(len(keys), 6, figsize=(15, 2.7 * len(keys)), squeeze=False)
for row, key in enumerate(keys):
    for col, rel in enumerate(random.sample(groups[key], min(6, len(groups[key])))):
        axes[row, col].imshow(Image.open(f"{OUT_DIR}/{rel}"), cmap="gray", vmin=0, vmax=255)
        axes[row, col].set_title(f"{key[0]} / {key[1]}\n{os.path.basename(rel)[:24]}", fontsize=7)
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.savefig("/kaggle/working/preview.png", dpi=80)
plt.show()